In [1]:
import pandas as pd
import os

This notebook is used to train simple classifiers using extracted geometric features from the hand-labeled dataset.

In [2]:
# load merged "env" data from hari
env_path = '/home/jko/ssl-cpi-analysis/data/from-hari-v2/clean_combined_campaign_env_data.csv'
df_env = pd.read_csv(env_path)
df_env.head()

,index,filename,date,Frame Width [pixels],Frame Height [pixels],Particle Width [micrometers],Particle Height [micrometers],Cutoff [%],Aggregate [%],Budding [%],...,Altitude [m],Pressure [hPa],Temperature [C],Ice Water Content [g/m3],PSD IWC [g/m3],concentration ratio,area ratio,mass ratio,Campaign,Perimeter [pixels]
0,1229,2000_0309_204605_250_2.png,2000-03-09 20:46:05,126.0,110.0,195.843,230.417,0.00,0.000,0.000,...,7617.879883,376.093658,-34.019299,0.041360,0.054664,0.000046,0.001051,0.003035,ARM,NaN
1,11589,2000_0313_190050_93_7.png,2000-03-13 19:00:50,43.0,60.0,87.400,50.600,0.00,0.000,0.000,...,8834.772461,315.023621,-46.836998,0.019560,0.049062,0.000000,0.000000,0.000000,ARM,NaN
2,2955,2000_0309_214537_826_5.png,2000-03-09 21:45:37,226.0,195.0,349.250,538.321,0.00,0.007,0.000,...,8692.738867,321.706464,-42.729707,0.054596,0.056262,0.000977,0.021351,0.024678,ARM,NaN
3,2470,2000_0309_213422_906_2.png,2000-03-09 21:34:22,233.0,298.0,756.057,360.955,4.61,0.002,0.000,...,9322.357031,292.931525,-48.294354,0.027008,0.049581,0.000039,0.000372,0.001016,ARM,NaN
4,18459,2000_0313_204457_990_26.png,2000-03-13 20:44:57,233.0,208.0,523.482,397.076,1.93,0.211,0.006,...,7011.962891,409.888617,-32.159441,0.063584,0.074550,0.000017,0.000230,0.000886,ARM,NaN


In [3]:
print(df_env.shape)
print(df_env['filename'].nunique())

(524033, 48)
524028


In [4]:
duplicate_rows = df_env[df_env.duplicated('filename', keep=False)]
duplicate_rows = duplicate_rows.sort_values(by='filename')
duplicate_rows

,index,filename,date,Frame Width [pixels],Frame Height [pixels],Particle Width [micrometers],Particle Height [micrometers],Cutoff [%],Aggregate [%],Budding [%],...,Altitude [m],Pressure [hPa],Temperature [C],Ice Water Content [g/m3],PSD IWC [g/m3],concentration ratio,area ratio,mass ratio,Campaign,Perimeter [pixels]
33933,13983,2002_0716_211311_476_15.png,2002-07-16 21:13:11,78.0,64.0,123.859,146.922,3.17,0.001,0.000,...,12519.719922,178.093878,-58.501601,0.089096,0.000000,0.000000,0.000000,0.000000,CRYSTAL_FACE_NASA,NaN
85638,155288,2002_0716_211311_476_15.png,2002-07-16 21:13:11,180.0,172.0,345.380,297.144,0.00,96.860,0.001,...,11930.247266,217.341470,-49.144500,0.217712,0.356982,0.000159,0.012172,0.021351,CRYSTAL_FACE_UND,NaN
34458,49423,2002_0729_192324_634_18.png,2002-07-29 19:23:24,40.0,44.0,79.076,40.345,1.79,0.000,0.000,...,12493.235547,178.838678,-59.475600,0.044775,0.072630,0.000000,0.000000,0.000000,CRYSTAL_FACE_NASA,NaN
129794,332739,2002_0729_192324_634_18.png,2002-07-29 19:23:24,171.0,132.0,183.569,338.378,0.00,59.673,0.000,...,10585.080078,262.326569,-40.357882,0.539192,1.280602,0.003854,0.367843,0.453456,CRYSTAL_FACE_UND,NaN
19551,49425,2002_0729_192324_634_21.png,2002-07-29 19:23:24,54.0,94.0,196.075,97.634,9.46,0.000,0.000,...,12493.235547,178.838678,-59.475600,0.044775,0.072630,0.000000,0.000000,0.000000,CRYSTAL_FACE_NASA,NaN
231085,332737,2002_0729_192324_634_21.png,2002-07-29 19:23:24,104.0,87.0,206.883,122.703,0.00,98.821,0.000,...,10585.080078,262.326569,-40.357882,0.539192,1.280602,0.003854,0.367843,0.453456,CRYSTAL_FACE_UND,NaN
37671,49443,2002_0729_192324_634_58.png,2002-07-29 19:23:24,64.0,62.0,107.483,130.744,7.14,0.000,0.000,...,12493.235547,178.838678,-59.475600,0.044775,0.072630,0.000000,0.000000,0.000000,CRYSTAL_FACE_NASA,NaN
280749,332734,2002_0729_192324_634_58.png,2002-07-29 19:23:24,105.0,94.0,148.149,189.281,0.00,0.003,0.000,...,10585.080078,262.326569,-40.357882,0.539192,1.280602,0.003854,0.367843,0.453456,CRYSTAL_FACE_UND,NaN
30928,49442,2002_0729_192324_634_62.png,2002-07-29 19:23:24,102.0,98.0,129.030,261.620,6.75,2.055,0.000,...,12493.235547,178.838678,-59.475600,0.044775,0.072630,0.000000,0.000000,0.000000,CRYSTAL_FACE_NASA,NaN
278483,332733,2002_0729_192324_634_62.png,2002-07-29 19:23:24,90.0,90.0,134.580,128.984,0.00,0.000,0.000,...,10585.080078,262.326569,-40.357882,0.539192,1.280602,0.003854,0.367843,0.453456,CRYSTAL_FACE_UND,NaN


In [5]:
duplicate_filenames = duplicate_rows['filename']

In [6]:
data_root = '/home/vanessa/hulk/cocpit/cpi_data/training_datasets/v3.1.0/hand_labeled_noaug'
# get a list of all filenames in this directory
png_filepaths = []
png_filenames = []

for root, dirs, files in os.walk(data_root):
    for file in files:
        if file.lower().endswith('.png'):
            png_filepaths.append(os.path.join(root, file))
            png_filenames.append(file)
print(png_filenames[:5])

['2016_1014_022252_52.png', '2004_1006_203416_99_10.png', '2004_1017_210056_693_56.png', '2016_1018_023545_68.png', '2004_1006_212858_501_16.png']


In [7]:
print(f'There are total {len(png_filenames)} labeled data.')

There are total 21537 labeled data.


In [8]:
# are any of the labeled data in the duplicate filenames list?
any(fname in duplicate_filenames.values for fname in png_filenames)

False

In [9]:
# how many of the labeled data are in env filenames?
png_set = set(png_filenames)
env_set = set(df_env['filename'])

matches = png_set.intersection(env_set)

In [10]:
len(matches)

1332

In [11]:
# get data from final predictions from vgg16 v3.1.0
import pandas as pd
from pathlib import Path

# Path to the folder containing CSV files
csv_dir = Path('/home/vanessa/hulk/cocpit/final_databases/vgg16/v3.1.0')

# Get list of all CSV files
csv_files = list(csv_dir.glob('*.csv'))

# Read and concatenate all CSVs
df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Save as Parquet file
save_dir = '/home/jko/ssl-cpi-analysis/data'
df_all.to_parquet(os.path.join(save_dir, 'predictions_vgg16_v3.1.0_all_campaigns.parquet'))

In [12]:
# how many of the labeled data are in final predictions data (vgg16, v3.1.0)
png_set = set(png_filenames)
env_set = set(df_all['filename'])

matches = png_set.intersection(env_set)
len(matches)

2405

In [13]:
df_all.columns

Index(['filename', 'date', 'frame width [pixels]', 'frame height [pixels]',
       'particle width [microns]', 'particle height [microns]', 'cutoff [%]',
       'aggregate [%]', 'budding rosette [%]', 'bullet rosette [%]',
       'column [%]', 'compact irregular [%]', 'fragment [%]',
       'planar polycrystal [%]', 'rimed [%]', 'sphere [%]', 'classification',
       'perim [pixels]', 'hull_area [pixels]', 'convex_perim [pixels]', 'blur',
       'contours [#]', 'contrast', 'cnt_area [pixels]', 'circularity',
       'solidity', 'complexity', 'equiv_d', 'phi', 'extreme_points',
       'filled_circular_area_ratio', 'roundness', 'perim_area_ratio',
       'perim [pixels].1', 'hull_area [pixels].1', 'convex_perim [pixels].1',
       'blur.1', 'contours [#].1', 'contrast.1', 'cnt_area [pixels].1',
       'circularity.1', 'solidity.1', 'complexity.1', 'equiv_d.1', 'phi.1',
       'extreme_points.1', 'filled_circular_area_ratio.1', 'roundness.1',
       'perim_area_ratio.1'],
      dtype='obje

In [15]:
# save png filepaths for labeled dataset as txt file
save_dir = '/home/jko/ssl-cpi-analysis/data'
save_filepath = os.path.join(save_dir, 'labeled_png_filepaths.txt')
with open(save_filepath, 'w') as f:
    for path in png_filepaths:
        f.write(f"{path}\n")

In [16]:
# Image processing functions
# 8 functions total: aspect ratio, elliptical aspect ratio, number of extreme points, contour area, area ratio, complexity, and circularity

'''
This script calculates:
(1) inputs/outputs necessary for supervised tabular ML 
(2) simultaneously, this dataframe can be used for deep learning training
- i.e., ID / output pairs 
'''

import cv2 as cv
import pandas as pd
import os, sys
import matplotlib.pyplot as plt
import numpy as np
import random


### ========= Helper functions ========= ###

def get_img(file_path):
    img = cv.imread(file_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2YCR_CB)[...,0]
    return img

def show_img(im, figsize=None, ax=None, alpha=None):
    if not ax: fig,ax = plt.subplots(figsize=figsize)
    ax.imshow(im, alpha=alpha, cmap=plt.cm.gray)
    ax.set_axis_off()
    return ax

def get_border(image, width):
    bg = np.zeros(image.shape)
    contours, _ = cv.findContours(image.copy(), cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    biggest = 0
    bigcontour = None
    for contour in contours:
        area = cv.contourArea(contour) 
        if area > biggest:
            biggest = area
            bigcontour = contour
    return cv.drawContours(bg, [bigcontour], 0, (255, 255, 255), width).astype(bool), contours 

def get_aspect_ratio(cnt):
    rect = cv.minAreaRect(cnt)
    # get length and width of contour
    x = rect[1][0]
    y = rect[1][1]
    rect_length = max(x, y)
    rect_width = min(x, y)
    phi = rect_width / rect_length
    return phi

def get_aspect_ratio_elip(cnt):
    if len(cnt) < 5:
        # fallback to bounding rectangle aspect ratio
        x, y, w, h = cv.boundingRect(cnt)
        if w == 0 or h == 0:
            return 0  # avoid division by zero or invalid shape
        # return ratio always <= 1 like in your original ellipse ratio
        return min(w, h) / max(w, h)
    else:
        ellipse = cv.fitEllipse(cnt)
        widthE = ellipse[1][0]
        heightE = ellipse[1][1]
        if widthE > heightE:
            phiE = heightE / widthE
        else:
            phiE = widthE / heightE
        return phiE

def get_extreme_pts(cnt):
    left = tuple(cnt[cnt[:, :, 0].argmin()][0])
    right = tuple(cnt[cnt[:, :, 0].argmax()][0])
    top = tuple(cnt[cnt[:, :, 1].argmin()][0])
    bottom = tuple(cnt[cnt[:, :, 1].argmax()][0])
    extreme_pts = np.std([left, right, top, bottom])
    return extreme_pts

def get_contour_area(cnt):
    area = cv.contourArea(cnt)
    return area

def get_contour_perimeter(cnt):
    perimeter = cv.arcLength(cnt, True)
    return perimeter

def get_min_circle(cnt):
    center ,radius = cv.minEnclosingCircle(cnt)
    perimeter_circle = 2*np.pi*radius
    area_circle = np.pi*(radius**2)
    return center, radius, perimeter_circle, area_circle

def get_area_ratio(cnt):
    area = get_contour_area(cnt)
    _,_,_,area_circle = get_min_circle(cnt)
    area_ratio = area/area_circle
    return area_ratio

def get_complexity(cnt):
    _, radius, _, _ = get_min_circle(cnt)
    area = get_contour_area(cnt)
    perimeter = get_contour_perimeter(cnt)
    Ac = np.pi * radius ** 2
    complexity = 10*(0.1-(area / (np.sqrt(area / Ac) * perimeter ** 2)))
    return complexity

def get_circularity(cnt):
    area = get_contour_area(cnt)
    perimeter = get_contour_perimeter(cnt)
    circularity = 4*np.pi*(area/(perimeter**2))
    return circularity

def process_img(path):
    img = get_img(path)
    _, contours = get_border(img, 5)
    cnt = contours[0]
    aspect_ratio = get_aspect_ratio(cnt)
    aspect_ratio_elip = get_aspect_ratio_elip(cnt)
    extreme_pts = get_extreme_pts(cnt)
    contour_area = get_contour_area(cnt)
    contour_perimeter = get_contour_perimeter(cnt)
    area_ratio = get_area_ratio(cnt)
    complexity = get_complexity(cnt)
    circularity = get_circularity(cnt)
    img_features = [aspect_ratio, aspect_ratio_elip, extreme_pts, \
        contour_area, contour_perimeter, area_ratio, \
        complexity, circularity]
    return img_features


In [17]:
# how long does it take to process 100 files?
files_subset = png_filepaths[:100] 
record_list = []
for f in files_subset:
    record = process_img(f)
    record_list.append(record)
df_labeled = pd.DataFrame(record_list)

In [16]:
import cv2
import pandas as pd
import os
import numpy as np

def get_img(file_path):
    return cv2.imread(file_path)

def get_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def get_thresh(gray, thresh_val=50):
    return cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY_INV)[1]

def find_largest_contour(thresh):
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, []
    largest = max(contours, key=cv2.contourArea)
    return largest, contours

def aspect_ratio_rect(cnt):
    rect = cv2.minAreaRect(cnt)
    x, y = rect[1][0], rect[1][1]
    rect_length = max(x, y)
    rect_width = min(x, y)
    return rect_width / rect_length if rect_length > 0 else np.nan

def aspect_ratio_ellipse(cnt):
    if len(cnt) >= 5:
        ellipse = cv2.fitEllipse(cnt)
        widthE, heightE = ellipse[1][0], ellipse[1][1]
        return min(widthE, heightE) / max(widthE, heightE) if max(widthE, heightE) > 0 else np.nan
    else:
        x, y, w, h = cv2.boundingRect(cnt)
        return min(w, h) / max(w, h) if max(w, h) > 0 else np.nan

def extreme_points_std(cnt):
    left = tuple(cnt[cnt[:, :, 0].argmin()][0])
    right = tuple(cnt[cnt[:, :, 0].argmax()][0])
    top = tuple(cnt[cnt[:, :, 1].argmin()][0])
    bottom = tuple(cnt[cnt[:, :, 1].argmax()][0])
    return np.std([left, right, top, bottom])

def contour_area(cnt):
    return cv2.contourArea(cnt)

def contour_perimeter(cnt):
    return cv2.arcLength(cnt, True)

def convex_hull(cnt):
    return cv2.convexHull(cnt)

def convex_perimeter(hull):
    return cv2.arcLength(hull, True)

def hull_area(hull):
    return cv2.contourArea(hull)

def min_enclosing_circle(cnt):
    _, radius = cv2.minEnclosingCircle(cnt)
    return radius

def filled_circular_area_ratio(area, radius):
    return area / (np.pi * radius ** 2) if radius > 0 else np.nan

def laplacian_var(gray):
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def circularity(area, perim):
    return (4.0 * np.pi * area) / (perim ** 2) if perim > 0 else np.nan

def roundness(area, convex_perim):
    return (4.0 * np.pi * area) / (convex_perim ** 2) if convex_perim > 0 else np.nan

def perim_area_ratio(perim, area):
    return perim / area if area > 0 else np.nan

def solidity(area, hull_area_val):
    return area / hull_area_val if hull_area_val > 0 else np.nan

def equiv_d(area):
    return np.sqrt(4 * area / np.pi) if area > 0 else np.nan

def complexity(area, perim, radius):
    Ac = np.pi * radius ** 2 if radius > 0 else np.nan
    if area > 0 and perim > 0 and Ac > 0:
        return 10 * (0.1 - (area / (np.sqrt(area / Ac) * perim ** 2)))
    else:
        return np.nan

def process_img(path):
    img = get_img(path)
    if img is None:
        return [path] + [np.nan]*13
    gray = get_gray(img)
    thresh = get_thresh(gray)
    cnt, contours = find_largest_contour(thresh)
    if cnt is None:
        return [path] + [np.nan]*13
    phi = aspect_ratio_rect(cnt)
    phi_elip = aspect_ratio_ellipse(cnt)
    extreme_pts = extreme_points_std(cnt)
    area = contour_area(cnt)
    perim = contour_perimeter(cnt)
    hull = convex_hull(cnt)
    convex_perim_val = convex_perimeter(hull)
    hull_area_val = hull_area(hull)
    radius = min_enclosing_circle(cnt)
    filled_circ_area_ratio = filled_circular_area_ratio(area, radius)
    lap_var = laplacian_var(gray)
    circ = circularity(area, perim)
    roundn = roundness(area, convex_perim_val)
    perim_area = perim_area_ratio(perim, area)
    solid = solidity(area, hull_area_val)
    equiv = equiv_d(area)
    complx = complexity(area, perim, radius)
    return [
        path, phi, phi_elip, extreme_pts, area, perim, filled_circ_area_ratio,
        complx, circ, roundn, perim_area, solid, equiv, lap_var
    ]



In [17]:
# how long does it take to process 100 files?
files_subset = png_filepaths[:100] 
record_list = []
for f in files_subset:
    record = process_img(f)
    record_list.append(record)
df_labeled = pd.DataFrame(record_list)

In [18]:
df_labeled.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.686028,0.654362,260.795628,322578.5,2705.948265,0.665657,0.460028,0.553613,0.882135,0.008388,0.960147,640.874171,21.106886
1,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.871030,0.873343,328.507515,525003.5,3957.369619,0.669537,0.590304,0.421268,0.818547,0.007538,0.889839,817.591106,9.312568
2,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.970808,0.925908,300.808909,492045.0,3625.362479,0.759562,0.570443,0.470448,0.869063,0.007368,0.907938,791.511940,29.683726
3,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.937285,0.694526,275.595265,367257.0,3029.019333,0.565226,0.467579,0.503009,0.811227,0.008248,0.933003,683.817326,10.816930
4,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.923469,0.746467,293.537476,397303.5,3310.468036,0.633394,0.544481,0.455568,0.804740,0.008332,0.880249,711.240133,53.601586


In [2]:
df_test = pd.read_parquet('/home/jko/ssl-cpi-analysis/data/labeled_data_geo_features.parquet')
df_test.head()

,classification,filename,aspect_ratio,aspect_ratio_elip,extreme_points,contour_area,contour_perimeter,filled_circular_area_ratio,complexity,circularity,roundness,perim_area_ratio,solidity,equiv_d,laplacian
0,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.686028,0.654362,260.795628,322578.5,2705.948265,0.665657,0.460028,0.553613,0.882135,0.008388,0.960147,640.874171,21.106886
1,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.871030,0.873343,328.507515,525003.5,3957.369619,0.669537,0.590304,0.421268,0.818547,0.007538,0.889839,817.591106,9.312568
2,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.970808,0.925908,300.808909,492045.0,3625.362479,0.759562,0.570443,0.470448,0.869063,0.007368,0.907938,791.511940,29.683726
3,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.937285,0.694526,275.595265,367257.0,3029.019333,0.565226,0.467579,0.503009,0.811227,0.008248,0.933003,683.817326,10.816930
4,compact_irreg,/home/vanessa/hulk/cocpit/cpi_data/training_da...,0.923469,0.746467,293.537476,397303.5,3310.468036,0.633394,0.544481,0.455568,0.804740,0.008332,0.880249,711.240133,53.601586


In [3]:
df_test.shape

(21537, 15)

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# remove any nans
df = df_test.dropna().reset_index(drop=True)

# train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
feature_cols = ['aspect_ratio', 'aspect_ratio_elip', 'extreme_points',
        'contour_area', 'contour_perimeter', 'filled_circular_area_ratio',
        'complexity', 'circularity', 'roundness', 'perim_area_ratio',
        'solidity', 'equiv_d', 'laplacian']
X_train = train_df[feature_cols]
y_train = train_df['classification']
X_test = test_df[feature_cols]
y_test = test_df['classification']

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression (no multi_class argument, higher max_iter)
clf = LogisticRegression(max_iter=2000, solver='lbfgs')
clf.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = clf.predict(X_test_scaled)

# Print classification report (zero_division=0 to silence warning)
print(classification_report(y_test, y_pred, zero_division=0))

# Compute and plot confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

NameError: name 'df_test' is not defined

In [20]:
import cv2

filepaths_txtfile = '/home/jko/ssl-cpi-analysis/data/labeled_png_filepaths.txt'

with open(filepaths_txtfile, 'r') as f:
    png_filepaths = [line.strip() for line in f if line.strip()]

for path in png_filepaths:
    img = cv2.imread(path)
    if img is None:
        print(f"Corrupted PNG: {path}")

Corrupted PNG: /home/vanessa/hulk/cocpit/cpi_data/training_datasets/v3.1.0/hand_labeled_noaug/compact_irreg/0419-200054_270_31.png


libpng error: IDAT: CRC error
